In [33]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [34]:
custos = pd.read_excel('exerc2.xlsx', sheet_name='custos')
demanda_produto = pd.read_excel('exerc2.xlsx', sheet_name='demanda_produto')
capacidade_producao = pd.read_excel('exerc2.xlsx', sheet_name='capacidade_producao')

In [35]:
custos = custos.set_index('produtos')
demanda_produto = demanda_produto.set_index('produtos')
capacidade_producao=capacidade_producao.set_index('fabricas')

In [36]:
dict_custos = custos.T.to_dict()
dict_custos

{'p1': {'f1': 31, 'f2': 29, 'f3': 32, 'f4': 28, 'f5': 29},
 'p2': {'f1': 45, 'f2': 41, 'f3': 46, 'f4': 42, 'f5': 43},
 'p3': {'f1': 38, 'f2': 35, 'f3': 40, 'f4': 0, 'f5': 0}}

In [37]:
dict_demanda_produto = demanda_produto['demanda'].to_dict()
dict_demanda_produto

{'p1': 1500, 'p2': 2500, 'p3': 2000}

In [38]:
dict_capacidade_producao = capacidade_producao['capacidade_producao'].to_dict()
dict_capacidade_producao

{'f1': 2000, 'f2': 1000, 'f3': 2000, 'f4': 1500, 'f5': 2500}

In [40]:
dict_capacidade_producao.keys()

dict_keys(['f1', 'f2', 'f3', 'f4', 'f5'])

In [52]:
model = pyo.ConcreteModel()

model.produtos = pyo.Set(initialize = dict_demanda_produto.keys())
model.fabricas = pyo.Set(initialize = dict_capacidade_producao.keys())
model.x = pyo.Var(model.produtos,model.fabricas, bounds=(0,None), domain=NonNegativeIntegers)

def objetivo(model):

    return sum(
        model.x[p,f] * dict_custos[p][f] for p in model.produtos for f in model.fabricas
    )
model.obj = pyo.Objective(rule=objetivo, sense=pyo.minimize)

def restricao_demanda_total_produtos(model,p):

    return sum(
        model.x[p,f] for f in model.fabricas
    ) == dict_demanda_produto[p] 
model.restricao_demanda_total_produtos = pyo.Constraint(model.produtos,rule=restricao_demanda_total_produtos)

def restricao_producao_fabricas(model, f):

    return sum(
        model.x[p,f] for p in model.produtos
    ) <= dict_capacidade_producao[f]
model.restricao_producao_fabricas = pyo.Constraint(model.fabricas, rule=restricao_producao_fabricas)

In [53]:
model.pprint()

2 Set Declarations
    fabricas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    5 : {'f1', 'f2', 'f3', 'f4', 'f5'}
    produtos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'p1', 'p2', 'p3'}

1 Var Declarations
    x : Size=15, Index=produtos*fabricas
        Key          : Lower : Value : Upper : Fixed : Stale : Domain
        ('p1', 'f1') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('p1', 'f2') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('p1', 'f3') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('p1', 'f4') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('p1', 'f5') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('p2', 'f1') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('p2', 'f2') :  

In [54]:
opt = SolverFactory('cplex')
reusltado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmpd821h31b.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpt80r14pc.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpt80r14pc.pyomo.lp
Objective sense      : Minimize
Variables            :      15  [General Integer: 15]
Objective nonzeros   :      13
Linear constraints   :       8  [Less: 5,  Equal: 3]
  Nonzeros           :      30
  RHS nonzeros       :       8

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 28.

In [55]:
model.x.pprint()

x : Size=15, Index=produtos*fabricas
    Key          : Lower : Value  : Upper : Fixed : Stale : Domain
    ('p1', 'f1') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('p1', 'f2') :     0 :    0.0 :  None : False : False : NonNegativeIntegers
    ('p1', 'f3') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('p1', 'f4') :     0 : 1500.0 :  None : False : False : NonNegativeIntegers
    ('p1', 'f5') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('p2', 'f1') :     0 : 1000.0 :  None : False : False : NonNegativeIntegers
    ('p2', 'f2') :     0 : 1000.0 :  None : False : False : NonNegativeIntegers
    ('p2', 'f3') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('p2', 'f4') :     0 :    0.0 :  None : False : False : NonNegativeIntegers
    ('p2', 'f5') :     0 :  500.0 :  None : False : False : NonNegativeIntegers
    ('p3', 'f1') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('p3', 'f2')

In [56]:
print("Custo total: ", pyo.value(model.obj))

Custo total:  149500.0
